# Assumption

The README did not specify if we are to use the SMTLib distribution for Z3. I am hence assuming that we are to use the python API [(documentation)](https://ericpony.github.io/z3py-tutorial/guide-examples.htm).

For me, it is easier to parse and has a cleaner syntax. I also used it for Assignment 1 and so have some level of familiarity with it. Python with jupyter notebooks also allows for easier writeups with code.



### Imports

In [ ]:
from z3 import Solver, Bool, And, Not, Or, Ints, Int, Implies, ForAll, Reals, sat
from sympy import symbols, sympify
from IPython.display import Markdown

### **Game of 21**

We can recursively understand the logic behind the model. 

- Firstly, note that a player who is left with a pile of size 1, 2, or 3, can always win.
    - Thus, $Win(1) = Win(2) = Win(3) = \top$
        - This is the base case, kind of like axioms

- Next, note that if a player has a winning position, it means that there is some removal choice they can make so that the resulting configation is not a winning postion. In other words, having a pile of size either 1 object, 2 objects or 3 objects lesser than the current pile size that leads to a loss.
    - So, $Win(i) = \neg Win(i-1) \lor \neg Win(i-2) \lor \neg Win(i-3)$
        - This is the recursive description, kind of like a rule of inference
        - $1 < n \leq 21$
        - The winning strategy would be to always pick this removal choice.

In [2]:
s = Solver()
Win = [Bool(f"Win({i})") for i in range(1,22)]

In [3]:
# base case
s.add(Win[0]==True, Win[1]==True, Win[2]==True)
# indexing is weird

s

[Win(1) == True, Win(2) == True, Win(3) == True]

In [4]:
# recursion
for i in range(3,21): # adjustment for zero indexing
    s.add(Win[i] == Or(Not(Win[i-1]), Not(Win[i-2]), Not(Win[i-3])))

s.push()
s

[Win(1) == True,
 Win(2) == True,
 Win(3) == True,
 Win(4) == Or(Not(Win(3)), Not(Win(2)), Not(Win(1))),
 Win(5) == Or(Not(Win(4)), Not(Win(3)), Not(Win(2))),
 Win(6) == Or(Not(Win(5)), Not(Win(4)), Not(Win(3))),
 Win(7) == Or(Not(Win(6)), Not(Win(5)), Not(Win(4))),
 Win(8) == Or(Not(Win(7)), Not(Win(6)), Not(Win(5))),
 Win(9) == Or(Not(Win(8)), Not(Win(7)), Not(Win(6))),
 Win(10) == Or(Not(Win(9)), Not(Win(8)), Not(Win(7))),
 Win(11) == Or(Not(Win(10)), Not(Win(9)), Not(Win(8))),
 Win(12) == Or(Not(Win(11)), Not(Win(10)), Not(Win(9))),
 Win(13) == Or(Not(Win(12)), Not(Win(11)), Not(Win(10))),
 Win(14) == Or(Not(Win(13)), Not(Win(12)), Not(Win(11))),
 Win(15) == Or(Not(Win(14)), Not(Win(13)), Not(Win(12))),
 Win(16) == Or(Not(Win(15)), Not(Win(14)), Not(Win(13))),
 Win(17) == Or(Not(Win(16)), Not(Win(15)), Not(Win(14))),
 Win(18) == Or(Not(Win(17)), Not(Win(16)), Not(Win(15))),
 Win(19) == Or(Not(Win(18)), Not(Win(17)), Not(Win(16))),
 Win(20) == Or(Not(Win(19)), Not(Win(18)), Not(Win(17))),
 Win(21) == Or(Not(Win(20)), Not(Win(19)), Not(Win(18)))]

- Finally, we check if the first player can win, knowing that they always start with a pile size of 21.
    - We want to show that the above setup leads to the conclusion $Win(21)$.
    - We add the negation of the conclusion to the premises and show that it is UNSAT.

In [5]:
s.add(Win[20] == False) # 20 because indexing

s.check()

unsat

The first player can thus always win. What about the winning strategy?

- Let's see if we can infer something by pushing $Win(21) = \top$ and getting the correponding valualtion (model)

In [6]:
s.pop()

s.add(Win[20] == True)

display(s.check())
s.model()

sat

[Win(1) = True,
 Win(4) = False,
 Win(9) = True,
 Win(17) = True,
 Win(13) = True,
 Win(20) = False,
 Win(14) = True,
 Win(15) = True,
 Win(21) = True,
 Win(3) = True,
 Win(16) = False,
 Win(18) = True,
 Win(19) = True,
 Win(5) = True,
 Win(8) = False,
 Win(10) = True,
 Win(6) = True,
 Win(11) = True,
 Win(12) = False,
 Win(7) = True,
 Win(2) = True]

- Let's observe the model more carefully:

In [7]:
for i in range(20,-1,-1):
    display(Markdown(f"For pile of size = {i+1}, can we make a winning choice, $ Win({i+1})$? $ \\ \\ {s.model()[Win[i]]}$"))

For pile of size = 21, can we make a winning choice, $ Win(21)$? $ \ \ True$

For pile of size = 20, can we make a winning choice, $ Win(20)$? $ \ \ False$

For pile of size = 19, can we make a winning choice, $ Win(19)$? $ \ \ True$

For pile of size = 18, can we make a winning choice, $ Win(18)$? $ \ \ True$

For pile of size = 17, can we make a winning choice, $ Win(17)$? $ \ \ True$

...

For pile of size = 5, can we make a winning choice, $ Win(5)$? $ \ \ True$

For pile of size = 4, can we make a winning choice, $ Win(4)$? $ \ \ False$

For pile of size = 3, can we make a winning choice, $ Win(3)$? $ \ \ True$

For pile of size = 2, can we make a winning choice, $ Win(2)$? $ \ \ True$

For pile of size = 1, can we make a winning choice, $ Win(1)$? $ \ \ True$

- Note that every multiple of 4 is false. This is precisesly the winning strategy. 


    - The first player can always force the second player into a multiple of 4. 
    - The second player can never force the first player into a multiple of 4.
    - Assuming optimal play by the first player, eventually, the second player will be forced into a state with a file size of 4.
        - They cannot remove all object from the pile here, so they remove 1, 2, or 3 onjects. 
        - Whatever their choice, the first player ends up in cases they necessarily win!

### **Non Linear Constraint Solving**

This is pretty straightforward, we just load the statements into a z3 solver.

In [8]:
x, y = Ints('x y')

In [9]:
type(x)

z3.z3.ArithRef

In [10]:
s = Solver()

In [11]:
s.add(
    x*x + y*y == 25,  # pyright: ignore[reportOperatorIssue]
    x + y == 7,
    x > 0,
    y > 0
)

s

[x*x + y*y == 25, x + y == 7, x > 0, y > 0]

In [12]:
s.check()

sat

While SAT, iterate append negation of solution to find new assignment, till UNSAT is reached. Display each solution as it comes.

In [13]:
while str(s.check()) != 'unsat':

    m = s.model()

    sol_x = int(str(m[x]))
    sol_y = int(str(m[y]))

    print("x =", sol_x, ", y =", sol_y)
    s.add(Not(And(x == sol_x, y == sol_y)))

x = 3 , y = 4
x = 4 , y = 3


$(x=3, y=4)$ and $(x=4,y=3)$ are the only integer solutions.

### **Invariant Synthesis Example**

- At the time of writing - it's almost the end of the semester and I'm running short on time. So I'm going to outline an approach that we'll reuse in Week 6 for invariant synthesis. Two birds one stone etc etc.
 
- [\[CVA03.pdf\]](..\week8\papers\CAV03.pdf) outlines Farkas' Lemma as follows:

**Theorem 1 (Farkas’ Lemma).** Consider the following system of linear inequalities over real-valued variables $( x_1, \ldots, x_n )$,

$$
S : 
\begin{bmatrix}
a_{11}x_1 + \cdots + a_{1n}x_n + b_1 \leq 0 \\
\vdots & \vdots & \vdots \\
a_{m1}x_1 + \cdots + a_{mn}x_n + b_m \leq 0
\end{bmatrix}
$$

When $S$ is satisfiable, it entails a given linear inequality  

$$
\psi : c_1x_1 + \cdots + c_nx_n + d \leq 0
$$

if and only if there exist non-negative real numbers $( \lambda_0, \lambda_1, \ldots, \lambda_m )$, such that  

$$
c_1 = \sum_{i=1}^m \lambda_i a_{i1}, \quad \cdots, \quad c_n = \sum_{i=1}^m \lambda_i a_{in}, \quad d = \left( \sum_{i=1}^m \lambda_i b_i \right) - \lambda_0
$$

Furthermore, $S$ is unsatisfiable if and only if the inequality $1 \leq 0$ can be derived as shown above.

- We specify this for generating invariants of the form $ax + by \leq c$ (equivalent to $ax + by - c \leq 0$, the sign of $c$ swapped)
    - Note that here, we assume there are only two program variables.
    - Any parameters passed to the program in the method call are treated as constants.

- Note that we have:

    - Base Case: 
    $$
        Inv(x_0,y_0)
    $$
    - Inductive Step:
    $$
        Inv(x_i,y_i) \land LoopCondition(x_i,y_i) \implies Inv(x_{i+1},y_{i+1})
    $$

- For our implementation, we put in the base case (initial states of the variables) into the solver as is.
- We then utilise Farkas' Lemma for the induction step. 

    - Note that $Inv(x_i,y_i)$, $LoopCondition(x_i,y_i)$ are linear inequalities. They also should entail a linear inequality, $Inv(x_{i+1},y_{i+1})$.
    - We write:

        $$
        S : 
        \begin{bmatrix}
        ax_i + by_i - c \leq 0\\
        l_1x_i + l_2y_i - l_3\leq 0
        \end{bmatrix}
        $$

        Where $l_1, l_2, l_3$ are coefficients of the the loop condition (which we assume to be **linear**)
        - This is kind of a necessary assumption. Because otherwise the only option would be to do bounded domain checking and insert all possible loop states (up to a point) into z3. This would mean the pipleine would have to **interpret** the dafny program pythonically, a significantly tougher task than reading some JSON. Hence, we stick with these assumptions.

        Our target linear inequality is:
        $$
        ax_{i+1} + by_{i+1} - c \leq 0
        $$

        Now, we can write the assignment variables $x_{i+1} , y_{i+1}$ as transitional constraints on $x,y$ (which we assume to be **linear**)
        - This is again kind of a necessary assumption, same reason as before.

        So, 

        $$
        x_{i+1} = t^x_i + t^x_2y_i + t^x_3
        $$
        $$
        y_{i+1} = t^y_1x_i + t^y_2y_i + t^y_3
        $$

        And our target inequality becomes:
        $$
        a(t^x_1x_i + t^x_2y_i + t^x_3) + b(t^y_1x_i + t^y_2y_i + t^y_3) - c \leq 0
        $$

        By expanding, grouping, and rearranging the terms, we can write:
        $$
        (at^x_1 + bt^y_1)x_i + (at^x_2 + bt^y_2)y_i + (at^x_3 + bt^y_3 - c)\leq 0
        $$

        And finally, by Farkas' Lemma: 
        
        $$
        \left( Inv(x_i,y_i) \land LoopCondition(x_i,y_i) \implies Inv(x_{i+1},y_{i+1}) \right) \iff \exists \lambda_0 \lambda_1, \lambda_2
        $$ 
        
        such that:
        $$
        (at^x_1 + bt^y_1) = \lambda_1a + \lambda_2l_1
        $$

        $$
        (at^x_2 + bt^y_2) = \lambda_1b + \lambda_2l_2
        $$

        $$
        (at^x_3 + bt^y_3 - c) = \lambda_1(-c) + \lambda_2(-l_3) - \lambda_0
        $$






- We were able to reduce our invariants down to an existential check over real numbers, optimizations for which are outlined in the paper and baked into z3. 

- Let's implement this and see what z3 does!

In [ ]:
# AI DISCLOSURE - Generated by DeepSeek for parsing
def parse_to_standard_form_simple(expr_str, variables):
    x, y = symbols('x y')
    
    expr = expr_str
    if variables:
        expr = expr.replace(variables[0], 'x')
    if len(variables) > 1:
        expr = expr.replace(variables[1], 'y')
    
    parsed = sympify(expr)
    
    if hasattr(parsed, 'lhs'):
        # Get inequality type
        if '<' in expr_str:
            if '<=' in expr_str:
                left_expr = parsed.lhs - parsed.rhs
            else:  # <
                left_expr = parsed.lhs - (parsed.rhs - 1)
            a = sympify(left_expr.coeff(x))
            b = sympify(left_expr.coeff(y)) if len(variables) > 1 else sympify(0)
            c = -left_expr.subs({x: 0, y: 0})
        else:  # > or >=
            if '>=' in expr_str:
                left_expr = parsed.rhs - parsed.lhs
            else:  # >
                left_expr = (parsed.rhs + 1) - parsed.lhs
            a = sympify(left_expr.coeff(x))
            b = sympify(left_expr.coeff(y)) if len(variables) > 1 else sympify(0)
            c = -left_expr.subs({x: 0, y: 0})
        
        # Convert to float if possible
        return (float(a) if a.is_number else a,
                float(b) if b.is_number else b,
                float(c) if c.is_number else c)
    
    raise ValueError("Invalid inequality")

tests = [
    ("i < 10", ["i"]),           # (1.0, 0.0, 9.0)
    ("i <= 10", ["i"]),          # (1.0, 0.0, 10.0)
    ("i > 5", ["i"]),            # (-1.0, 0.0, -6.0)
    ("i >= 5", ["i"]),           # (-1.0, 0.0, -5.0)
    ("a < b + 5", ["a", "b"]),   # (1.0, -1.0, 5.0)
    ("2*x >= y - 3", ["x", "y"]), # (-2.0, 1.0, -3.0)
]

for expr, vars_list in tests:
    a, b, c = parse_to_standard_form_simple(expr, vars_list)
    print(f"{expr:20} -> ({a}, {b}, {c})")

i < 10               -> (1.0, 0.0, 9.0)
i <= 10              -> (1.0, 0.0, 10.0)
i > 5                -> (-1.0, 0.0, -6.0)
i >= 5               -> (-1.0, 0.0, -5.0)
a < b + 5            -> (1.0, -1.0, 4.0)
2*x >= y - 3         -> (-2.0, 1.0, 3.0)


In [ ]:
variables = ['i', 'sum']
parameters = ['n']

In [19]:
states = []
i = 0
sum = 0
n = 10

while i < n:
    states.append((i, sum))
    sum = sum + i
    i = i + 1

states

[(0, 0),
 (1, 0),
 (2, 1),
 (3, 3),
 (4, 6),
 (5, 10),
 (6, 15),
 (7, 21),
 (8, 28),
 (9, 36)]

In [20]:
def linear_invariant(i, sum):
    return a*i + b*sum + c <= 0

def loop_condition(i, n):
    return i < n

In [21]:
s = Solver()

s.add(
    # generated invariant should hold at initialization
    linear_invariant(i=0, sum=0),
)

for i, sum in states:
    s.add(
        Implies(
            And(loop_condition(i, n), linear_invariant(i, sum)),
            # if loop condition holds and invariant holds before loop body executes

            # ===>

            # invariant should also hold after loop body
            linear_invariant(i+1, sum+i)
        )
    )

s

[False,
 Implies(And(True, False), False),
 Implies(And(True, False), True),
 Implies(And(True, True), True),
 Implies(And(True, True), False),
 Implies(And(True, False), False),
 Implies(And(True, False), False),
 Implies(And(True, False), False),
 Implies(And(True, False), False),
 Implies(And(True, False), False),
 Implies(And(True, False), False)]

In [22]:
λ1 = 1

In [ ]:
from typing